## **ETL Walmart Sales**

In [66]:
import pandas as pd

In [67]:
walmart_df = pd.read_csv("walmart_data.csv")
walmart_df.head(5) 

,Order ID,Order Date,Ship Date,Customer Name,Country,City,State,Category,Product Name,Sales,Quantity,Profit
0,CA-2013-138688,13/06/2013,17/06/2013,Darrin Van Huff,United States,Los Angeles,California,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,6.87
1,CA-2011-115812,09/06/2011,14/06/2011,Brosina Hoffman,United States,Los Angeles,California,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.86,7,14.17
2,CA-2011-115812,09/06/2011,14/06/2011,Brosina Hoffman,United States,Los Angeles,California,Art,Newell 322,7.28,4,1.97
3,CA-2011-115812,09/06/2011,14/06/2011,Brosina Hoffman,United States,Los Angeles,California,Phones,Mitel 5320 IP Phone VoIP phone,907.15,4,90.72
4,CA-2011-115812,09/06/2011,14/06/2011,Brosina Hoffman,United States,Los Angeles,California,Binders,DXL Angle-View Binders with Locking Rings by S...,18.50,3,5.78


In [68]:
# Información del dataframe
walmart_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3203 entries, 0 to 3202
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order ID       3203 non-null   object 
 1   Order Date     3203 non-null   object 
 2   Ship Date      3203 non-null   object 
 3   Customer Name  3203 non-null   object 
 4   Country        3203 non-null   object 
 5   City           3203 non-null   object 
 6   State          3203 non-null   object 
 7   Category       3203 non-null   object 
 8   Product Name   3203 non-null   object 
 9   Sales          3203 non-null   float64
 10  Quantity       3203 non-null   int64  
 11  Profit         3203 non-null   float64
dtypes: float64(2), int64(1), object(9)
memory usage: 300.4+ KB


In [69]:
# No tiene filas duplicadas puesto que las filas y columnas siguen siendo las mismas después de drop_duplicates
walmart_df.drop_duplicates(keep='first')
walmart_df.shape

(3203, 12)

**Observaciones:** Este dataframe tiene 3203 filas y 12 columnas sin valores nulos ni duplicados, Order Date y Ship Date estan en tipo de dato Object por lo que transformaremos a tipo de dato fecha con formato de MySQL (YYYY-MM-DD).

In [70]:
# Verificamos que tipo de formato tienen las fechas
walmart_df[['Order Date','Ship Date']].head()

,Order Date,Ship Date
0,13/06/2013,17/06/2013
1,09/06/2011,14/06/2011
2,09/06/2011,14/06/2011
3,09/06/2011,14/06/2011
4,09/06/2011,14/06/2011


In [71]:
# Definimos la función para convertir fechas
def convertir_fechas(columna1, columna2):
    walmart_df[columna1] = pd.to_datetime(walmart_df[columna1], format='%d/%m/%Y')
    walmart_df[columna2] = pd.to_datetime(walmart_df[columna2], format='%d/%m/%Y')

# Llamamos a la función
convertir_fechas('Order Date', 'Ship Date')  

# Corroboramos los cambios
print(walmart_df[['Order Date','Ship Date']].head())
print()
print(walmart_df[['Order Date','Ship Date']].dtypes)

  Order Date  Ship Date
0 2013-06-13 2013-06-17
1 2011-06-09 2011-06-14
2 2011-06-09 2011-06-14
3 2011-06-09 2011-06-14
4 2011-06-09 2011-06-14

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object


### **Granularidad del dataset**
Ahora miremos las estadisticas basicas de los atributos tanto de tipo númerico, sting y object para analizar las relaciones existentes.

In [72]:
# Veamos las estadisticas basicas para las columnas del tipo numérico
walmart_df.describe()

,Order Date,Ship Date,Sales,Quantity,Profit
count,3203,3203,3203.000000,3203.000000,3203.000000
mean,2013-05-10 03:06:07.530440192,2013-05-14 01:25:25.195129600,226.493266,3.828910,33.849138
min,2011-01-07 00:00:00,2011-01-09 00:00:00,0.990000,1.000000,-3399.980000
25%,2012-05-22 00:00:00,2012-05-26 00:00:00,19.440000,2.000000,3.850000
50%,2013-07-22 00:00:00,2013-07-25 00:00:00,60.840000,3.000000,11.170000
75%,2014-05-23 00:00:00,2014-05-27 00:00:00,215.810000,5.000000,33.000000
max,2014-12-31 00:00:00,2015-01-06 00:00:00,13999.960000,14.000000,6719.980000
std,NaN,NaN,524.876911,2.260947,174.109155


In [73]:
walmart_df.select_dtypes(include=['object','string']).describe()

,Order ID,Customer Name,Country,City,State,Category,Product Name
count,3203,3203,3203,3203,3203,3203,3203
unique,1611,686,1,169,11,17,1494
top,CA-2013-165330,William Brown,United States,Los Angeles,California,Binders,Staples
freq,11,24,3203,747,2001,471,60


De la descripción anterior podemos observar los siguientes puntos:
1. Order ID no se puede usar como índice unico ya que tiene duplicados, ya que cada fila representa una linea de pedido y no un pedido completo. 
2. Los clientes pueden tener asociados más de un Order ID.
3. Los datos son solo de los Estados Unidos.
4. Debería existir una única fecha de pedido por Order ID y para Ship Date.
5. Cada producto debe pertenecer a una sola categoría.
6. Cada órden debe tener un solo cliente asociado.
7. Cada órden debe tener una sola fecha de Order Date y Ship Date.

In [74]:
# Definimos una función que nos de los valores unicos de columns2 para los datos agrupados por columna1
def validacion1(columna1, columna2):
    return(
        walmart_df.groupby(columna1)[columna2].nunique().loc[lambda x: x > 1].reset_index()
    )

In [75]:
# Verificamos si cada producto pertenece a una y solo una categoria
validacion1("Product Name", "Category")

,Product Name,Category
0,Staples,10


In [76]:
# Verificamos si las ciudades perteneces a un y solo un estado
validacion1("City", "State")

,City,State
0,Redmond,2


In [77]:
# Verificamos si cada orden pertenece a un y solo un cliente
validacion1("Order ID", "Customer Name")

,Order ID,Customer Name


In [78]:
# Verificamos si la orden tiene asociada una y solo una fecha de orden
validacion1("Order ID", "Order Date")

,Order ID,Order Date


**De lo anterior se observa que:**

1. Normalización de categorías de productos: El producto "Staples" se encuentra clasificado erróneamente en 10 categorías distintas, lo que evidencia una inconsistencia en la calidad de los datos. Dado que este comportamiento se repite en otros artículos, se implementará un proceso de unificación para homologar cada producto bajo una única categoría estratégica.

2. Resolución de ambigüedad geográfica: Se identificaron registros de ciudades homónimas ubicadas en estados diferentes. Para garantizar la integridad del análisis y evitar duplicidades, se creará una columna calculada que concatene los campos de "Estado" y "Ciudad", generando así un identificador geográfico único.

3. Validación de la granularidad de las órdenes: El análisis confirma que cada "Order ID" está asociado estrictamente a un único cliente y a una sola fecha de compra. Este comportamiento alinea los registros con las reglas de negocio establecidas y ratifica la consistencia y fiabilidad de la base de datos.

In [79]:
# Revisamos las diferentes categorias asociadas a Staples
categories_para_Staples = walmart_df.loc[walmart_df['Product Name'] == 'Staples', 'Category'].unique()
pd.DataFrame(categories_para_Staples, columns=['Category'])

,Category
0,Envelopes
1,Paper
2,Fasteners
3,Supplies
4,Labels
5,Art
6,Furnishings
7,Binders
8,Appliances
9,Storage


In [83]:
# Cambiamos a 'fasteners' todas las clasificaciones para 'Staples' y posteriormente comprobamos el cambio
walmart_df.loc[walmart_df['Product Name'] == 'Staples','Category'] = 'Fasteners'
walmart_df[walmart_df['Product Name'] == 'Staples']['Category'].unique()

array(['Fasteners'], dtype=object)

In [84]:
# Revisamos los estados que tiene la misma ciudad Redmond
Estados_para_Redmond = walmart_df.loc[walmart_df['City'] == 'Redmond', 'State'].unique()
pd.DataFrame(Estados_para_Redmond, columns=['Category'])

,Category
0,Oregon
1,Washington


hecho lo anterior ya podemos considerar que la granularidad del dataset es buena y podemos proceder a exportar a un csv

In [85]:
walmart_df.to_csv('walmart_sales_cleaned.csv', index=True)